In [15]:
# ============================================================
# 1. Install Dependencies
# ============================================================

!pip install -q -U transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 32.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 43.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [1]:

# ============================================================
# 2. Imports
# ============================================================

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)




In [5]:
!pip show torch transformers datasets peft accelerate bitsandbytes

Name: torch
Version: 2.9.0+cu126
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, easyocr, fastai, kornia, peft, pytorch-ignite, pytorch-lightning, sentence-transformers, timm, torchaudio, torchdata, torchmetrics, torchvision
---
Name: transformers
Version: 5.2.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in t

In [17]:
import os
os.listdir("/kaggle/input/datasets/venkatamanojkammara/data-v3")

['v3_training.jsonl', 'v3_validation.jsonl']

In [18]:
# ============================================================
# 3. Load Dataset
# ============================================================

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/venkatamanojkammara/data-v3/v3_training.jsonl",
        "validation": "/kaggle/input/datasets/venkatamanojkammara/data-v3/v3_validation.jsonl"
    }
)

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['sample_id', 'prompt', 'completion', 'text'],
        num_rows: 525
    })
    validation: Dataset({
        features: ['sample_id', 'prompt', 'completion', 'text'],
        num_rows: 54
    })
})


In [19]:
# ============================================================
# 4. Load Phi-2 Model
# ============================================================

model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Phi-2 needs manual pad token
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [20]:
# ============================================================
# 5. Apply LoRA
# ============================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 5,242,880 || all params: 2,784,926,720 || trainable%: 0.1883


In [21]:
# ============================================================
# 6. Tokenization
# ============================================================

MAX_LENGTH = 768


def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


tokenized_dataset = dataset.map(
    tokenize_function,
    remove_columns=["text"]
)

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

In [22]:
# ============================================================
# 7. Data Collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [24]:
# ============================================================
# 8. Training Arguments
# ============================================================

training_args = TrainingArguments(

    output_dir="/kaggle/working/phi2-transmission-model",

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    fp16=True,

    report_to="none",

    gradient_checkpointing=True
)

In [26]:
# ============================================================
# 9. Trainer
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator
)

In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.751021,0.633296
2,0.545823,0.530267
3,0.510062,0.506283


TrainOutput(global_step=99, training_loss=0.7485762894755662, metrics={'train_runtime': 1610.48, 'train_samples_per_second': 0.978, 'train_steps_per_second': 0.061, 'total_flos': 1.9260616015872e+16, 'train_loss': 0.7485762894755662, 'epoch': 3.0})

In [28]:
# ============================================================
# 11. Save Model
# ============================================================

trainer.save_model("/kaggle/working/phi2-transmission-model")

tokenizer.save_pretrained("/kaggle/working/phi2-transmission-model")

('/kaggle/working/phi2-transmission-model/tokenizer_config.json',
 '/kaggle/working/phi2-transmission-model/tokenizer.json')

In [29]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch


In [30]:
base_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    torch_dtype=torch.float16,
    device_map="auto"
)



Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [31]:
model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/phi2-transmission-model"
)

In [32]:
tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/working/phi2-transmission-model"
)

In [37]:
prompt = """
You are an automotive transmission control software engineer.

Your task is to derive detailed L4 transmission software requirements from higher-level requirements.

L4 requirements represent software-level logical rules that define how the Transmission Control Unit (TCU) should behave under specific conditions.

---------------------------------------------------------------------

OUTPUT RULES

Follow these rules strictly:

1. Generate logical software requirements only.
2. Each requirement must follow the format:

   IF <condition> THEN <action>

3. Strictly Generate L4 requirements only.
4. Number each requirement starting from 1.
5. Each requirement must appear on a new line.
6. Use transmission-related signals and variables where appropriate.
7. Do NOT generate explanations.
8. Do NOT generate any markers such as:
   ### END
   ### MODE
   ### CODE
   ### L4
   or similar tokens.

Only generate the numbered requirements.

---------------------------------------------------------------------

EXAMPLE

L2 Requirement:
The transmission shall prevent gear shift when the brake pedal is pressed.

L3 Requirement:
The TCU shall inhibit gear shift when Brake_Pedal = Pressed.

L4 Requirements:

1. IF Brake_Pedal = Pressed THEN Shift_Request = Rejected
2. IF Shift_Request = Rejected THEN Maintain_Current_Gear
3. IF Brake_Pedal = Released THEN Enable_Shift_Request
4. IF Brake_Pedal = Pressed THEN Log_Shift_Inhibition_Event
5. IF Brake_Pedal = Pressed THEN Disable_Shift_Actuator

---------------------------------------------------------------------

TASK

L2 Requirement:
The transmission shall prevent reverse gear engagement at high speed.

L3 Requirement:
The TCU shall reject reverse gear request when vehicle speed exceeds limit.

L4 Requirements:

1.
"""

In [38]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.3,
    top_p=0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



You are an automotive transmission control software engineer.

Your task is to derive detailed L4 transmission software requirements from higher-level requirements.

L4 requirements represent software-level logical rules that define how the Transmission Control Unit (TCU) should behave under specific conditions.

---------------------------------------------------------------------

OUTPUT RULES

Follow these rules strictly:

1. Generate logical software requirements only.
2. Each requirement must follow the format:

   IF <condition> THEN <action>

3. Strictly Generate L4 requirements only.
4. Number each requirement starting from 1.
5. Each requirement must appear on a new line.
6. Use transmission-related signals and variables where appropriate.
7. Do NOT generate explanations.
8. Do NOT generate any markers such as:
   ### END
   ### MODE
   ### CODE
   ### L4
   or similar tokens.

Only generate the numbered requirements.

---------------------------------------------------------

In [35]:
import shutil

shutil.make_archive(
    "/kaggle/working/phi2-transmission-model",
    "zip",
    "/kaggle/working/phi2-transmission-model"
)

'/kaggle/working/phi2-transmission-model.zip'